# Monga Qwen3 0.6B — QLoRA SFT smoke run

This notebook trains the first narrow Monga memory-grounding adapter on the synthetic seed dataset.

**Colab:** Runtime → Change runtime type → GPU before running the cells.


In [ ]:
!nvidia-smi


In [ ]:
!pip install -q -U unsloth trl datasets accelerate


In [ ]:
import os, shutil, pathlib
repo_dir = pathlib.Path('/content/Monga')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
!git clone -q -b feat/monga-finetune-sft-seed https://github.com/joker-bot0420/Monga.git /content/Monga
%cd /content/Monga
!git rev-parse --abbrev-ref HEAD
!git rev-parse --short HEAD


In [ ]:
!python finetune/validate_dataset.py


## Train

This is intentionally a tiny smoke run: Qwen3-0.6B, 4-bit QLoRA, 3 epochs, 1024-token context.
The held-out set is not used for gradient updates.


In [ ]:
!python finetune/train_sft.py \
  --model Qwen/Qwen3-0.6B \
  --output-dir /content/monga-qwen3-0.6b-memory-sft \
  --epochs 3 \
  --max-seq-length 1024 \
  --batch-size 1 \
  --grad-accum 4


In [ ]:
from pathlib import Path
adapter = Path('/content/monga-qwen3-0.6b-memory-sft/adapter')
assert adapter.exists(), f'Adapter not found: {adapter}'
print('Adapter files:')
for p in sorted(adapter.iterdir()):
    print(f'  {p.name}: {p.stat().st_size / (1024*1024):.2f} MB')


In [ ]:
!cd /content/monga-qwen3-0.6b-memory-sft && zip -qr /content/monga-qwen3-0.6b-memory-sft-adapter.zip adapter
from google.colab import files
files.download('/content/monga-qwen3-0.6b-memory-sft-adapter.zip')


## Stop here after the first run

Do not merge/export to GGUF yet. Bring back the training output (especially train/eval loss and any error) so we can verify the smoke run before evaluating held-out behavior.
